# 1. Setup

In [ ]:
import sys
import time
from collections import defaultdict
from pathlib import Path

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error as MAE

In [ ]:
PROJECT_ROOT = Path().resolve().parent
PARQUET_DATA_PATH = "../data/parquet_data/"
sys.path.append(str(PROJECT_ROOT))

In [ ]:
from utils.data_pipeline import DataPipeline
from utils.feature_engineering import get_month_splits
from utils.loading import load_parquet_data
from utils.model_pipeline import load_or_train_sklearn

In [ ]:
float_formatter = lambda x: (
    f"{x:.2e}" if abs(x) < 0.01 and x != 0 else f"{x:.2f}"
)
pd.set_option(
    "display.float_format",
    float_formatter,
    "display.max_rows",
    200,
    "display.max_columns",
    100,
)
sns.set_style("whitegrid")

In [ ]:
SEGMENT_F = ["county", "product_type", "is_business"]
CATEGORICAL_F = SEGMENT_F + ["is_consumption"]
RECORD_F = CATEGORICAL_F + ["datetime"]
RAND = 10

# 2. Data Preparation

In [ ]:
LAG_SPECS = {
    "2d": {
        "6h": ["mean"],
        "12h": ["mean"],
        "24h": ["mean"],
        "168h": ["mean"],
    },
    "3d": {
        "6h": ["mean"],
        "12h": ["mean"],
    },
    "7d": {
        "12h": ["mean"],
        "24h": ["mean"],
    },
}

In [ ]:
dp = DataPipeline(PARQUET_DATA_PATH)

In [ ]:
dp.load()

In [ ]:
dp.prepare()

In [ ]:
dp.merge()

In [ ]:
dp.transform_datetime()

In [ ]:
dp.add_lags_and_rollings("target", LAG_SPECS, "datetime", CATEGORICAL_F)

In [ ]:
dp.add_ratio_features()

In [ ]:
df = dp.df

# 3. Baseline

## Train-Test Splits

In [ ]:
start_ts = df["datetime"].min()
end_ts = df["datetime"].max()
print(
    f"First timestamp: {start_ts}",
    f"Last timestamp: {end_ts}",
    f"Time range: {end_ts - start_ts}",
    sep="\n",
)

First timestamp: 2021-09-01 00:00:00
Last timestamp: 2023-05-31 23:00:00
Time range: 637 days 23:00:00


In [ ]:
splits = get_month_splits(start_ts, 16, 1, 1, 5)
splits_val = splits[:3]
splits_test = splits[3:]
print("Validation", *splits_val, sep="\n")
print("Test", *splits_test, sep="\n")

Validation
{'train': (Timestamp('2021-09-01 00:00:00'), Timestamp('2022-12-31 23:00:00')), 'test': (Timestamp('2023-01-01 00:00:00'), Timestamp('2023-01-31 23:00:00'))}
{'train': (Timestamp('2021-09-01 00:00:00'), Timestamp('2023-01-31 23:00:00')), 'test': (Timestamp('2023-02-01 00:00:00'), Timestamp('2023-02-28 23:00:00'))}
{'train': (Timestamp('2021-09-01 00:00:00'), Timestamp('2023-02-28 23:00:00')), 'test': (Timestamp('2023-03-01 00:00:00'), Timestamp('2023-03-31 23:00:00'))}
Test
{'train': (Timestamp('2021-09-01 00:00:00'), Timestamp('2023-03-31 23:00:00')), 'test': (Timestamp('2023-04-01 00:00:00'), Timestamp('2023-04-30 23:00:00'))}
{'train': (Timestamp('2021-09-01 00:00:00'), Timestamp('2023-04-30 23:00:00')), 'test': (Timestamp('2023-05-01 00:00:00'), Timestamp('2023-05-31 23:00:00'))}


## Training

In [ ]:
COLUMNS_TO_DROP = ["datetime"]

In [ ]:
cat_cols = df.select_dtypes(include=["bool", "category"]).columns.to_list()
cat_cols

['county',
 'product_type',
 'is_business',
 'is_consumption',
 'national_holiday',
 'observance_day',
 'season_event',
 'dst',
 'hour',
 'day_of_week',
 'month',
 'quarter']

In [ ]:
notbus_prod_config = {
    "model_label": "lgbm_mae",
    "model_cls": LGBMRegressor,
    "eval_week": True,
    "early_stopping_rounds": 100,
    "model_params": {
        "n_estimators": 1000,
        "learning_rate": 0.1,
        "max_depth": 0,
        "num_leaves": 13,
        "random_state": RAND,
        "min_child_samples": 100,
        "reg_alpha": 0.1,
        "reg_lambda": 1,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "objective": "regression_l1",
        "metric": "mae",
        "n_jobs": -1,
        "force_col_wise": True,
        "verbosity": -1,
        "device": "cpu",
    },
}
notbus_cons_config = {
    "model_label": "lgbm_mae",
    "model_cls": LGBMRegressor,
    "eval_week": True,
    "early_stopping_rounds": 100,
    "model_params": {
        "n_estimators": 1000,
        "learning_rate": 0.1,
        "max_depth": 0,
        "num_leaves": 13,
        "random_state": RAND,
        "min_child_samples": 400,
        "reg_alpha": 1,
        "reg_lambda": 5,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "objective": "regression_l1",
        "metric": "mae",
        "n_jobs": -1,
        "force_col_wise": True,
        "verbosity": -1,
        "device": "cpu",
    },
}
bus_prod_config = {
    "model_label": "lgbm_mae",
    "model_cls": LGBMRegressor,
    "eval_week": True,
    "early_stopping_rounds": 100,
    "model_params": {
        "n_estimators": 1000,
        "learning_rate": 0.1,  # 0.15
        "max_depth": 0,
        "num_leaves": 13,
        "random_state": RAND,
        "min_child_samples": 100,
        "reg_alpha": 0.1,
        "reg_lambda": 1,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "objective": "regression_l1",
        "metric": "mae",
        "n_jobs": -1,
        "force_col_wise": True,
        "verbosity": -1,
        "device": "cpu",
    },
}
bus_cons_config = {
    "model_label": "lgbm_mse",
    "model_cls": LGBMRegressor,
    "eval_week": True,
    "early_stopping_rounds": 100,
    "model_params": {
        "n_estimators": 1000,
        "learning_rate": 0.1,
        "max_depth": 0,
        "num_leaves": 13,
        "random_state": RAND,
        "min_child_samples": 500,
        "reg_alpha": 2,
        "reg_lambda": 10,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "objective": "mean_squared_error",
        "metric": "mae",
        "n_jobs": -1,
        "force_col_wise": True,
        "verbosity": -1,
        "device": "cpu",
    },
}
configs = {
    (False, False): notbus_prod_config,
    (False, True): notbus_cons_config,
    (True, False): bus_prod_config,
    (True, True): bus_cons_config,
}

In [ ]:
baseline_preds = []
baseline_scores = []
baseline_ev_his = defaultdict(dict)
baseline_fi = []
actual_iters = []

for i, split in enumerate(splits_val):
    for is_cons in df["is_consumption"].unique():
        is_consumption_label = "consumption" if is_cons else "production"

        for is_bus in df["is_business"].unique():
            is_business_label = "business" if is_bus else "not_business"
            purpose = (
                f"lgbm_baseline_{is_consumption_label}_{is_business_label}"
            )

            df_subset = df[
                (df["is_consumption"] == is_cons)
                & (df["is_business"] == is_bus)
            ]
            test_mask = df_subset["datetime"].between(
                split["test"][0], split["test"][1]
            )

            train_cols = df_subset.columns.difference(
                COLUMNS_TO_DROP + ["target"], sort=False
            )

            X_test = df_subset.loc[test_mask, train_cols]
            y_test = df_subset.loc[test_mask, "target"]

            config = configs[(is_bus, is_cons)]

            start_time = time.perf_counter()

            model, was_trained, history = load_or_train_sklearn(
                models_dir="../models",
                notebook="feature_engineering",
                purpose=purpose,
                model_label=config["model_label"],
                model_cls=config["model_cls"],
                model_params=config["model_params"],
                split=split,
                df=df_subset,
                target_col="target",
                drop_cols=COLUMNS_TO_DROP,
                cat_cols=cat_cols,
                eval_week=config["eval_week"],
                early_stopping_rounds=config["early_stopping_rounds"],
                random_state=RAND,
            )

            elapsed_time = time.perf_counter() - start_time
            print(
                f"split {i}",
                f"{purpose}",
                # f"{model_label}",
                f"train time: {elapsed_time:.2f}s",
                sep=" | ",
            )
            baseline_ev_his[i][purpose] = history

            test_preds = model.predict(X_test)
            test_mae = MAE(y_test, test_preds)

            train_mask = df_subset["datetime"].between(
                split["train"][0],
                (
                    (split["train"][1] - pd.Timedelta("1W"))
                    if config["early_stopping_rounds"]
                    else split["train"][1]
                ),
            )  # exclude eval period
            X_train = df_subset.loc[train_mask, train_cols]
            y_train = df_subset.loc[train_mask, "target"]
            train_preds = model.predict(X_train)
            train_mae = MAE(y_train, train_preds)

            baseline_scores.append(
                {
                    "split": i,
                    # "purpose": purpose,
                    "is_business": is_business_label,
                    "is_consumption": is_consumption_label,
                    "train_mae": train_mae,
                    "test_mae": test_mae,
                }
            )
            print(
                f"Train MAE: {train_mae:.2f}",
                f"Test MAE: {test_mae:.2f}",
                f"Ratio: {test_mae / train_mae:.2f}",
                sep=" | ",
            )

            fold_df = pd.DataFrame(
                {
                    # "model_label": model_label,
                    "split": i,
                    "is_business": X_test["is_business"],
                    "is_consumption": X_test["is_consumption"],
                    "target": y_test,
                    "preds": test_preds,
                },
                index=X_test.index,
            )
            fold_df["abs_e"] = (fold_df["target"] - fold_df["preds"]).abs()
            baseline_preds.append(fold_df)

            meta = pd.DataFrame(
                [[i, f"{is_business_label}_{is_consumption_label}"]],
                columns=["split", "segment"],
            )
            fi = pd.DataFrame(
                [model.booster_.feature_importance(importance_type="gain")],
                columns=model.feature_name_,
            ).drop(columns=["is_business", "is_consumption"])
            fi = pd.concat([meta, fi], axis=1)
            baseline_fi.append(fi)

            actual_iters.append(
                {
                    "split": i,
                    "is_bus": is_bus,
                    "is_cons": is_cons,
                    "best_iter": model.booster_.best_iteration,
                }
            )

split 0 | lgbm_baseline_production_not_business | train time: 7.65s
Train MAE: 27.53 | Test MAE: 4.17 | Ratio: 0.15
split 0 | lgbm_baseline_production_business | train time: 10.98s
Train MAE: 27.44 | Test MAE: 3.32 | Ratio: 0.12
split 0 | lgbm_baseline_consumption_not_business | train time: 35.26s
Train MAE: 8.96 | Test MAE: 26.59 | Ratio: 2.97
split 0 | lgbm_baseline_consumption_business | train time: 6.92s
Train MAE: 89.35 | Test MAE: 102.93 | Ratio: 1.15
split 1 | lgbm_baseline_production_not_business | train time: 23.38s
Train MAE: 24.81 | Test MAE: 17.47 | Ratio: 0.70
split 1 | lgbm_baseline_production_business | train time: 7.69s
Train MAE: 30.63 | Test MAE: 9.81 | Ratio: 0.32
split 1 | lgbm_baseline_consumption_not_business | train time: 35.10s
Train MAE: 9.46 | Test MAE: 24.91 | Ratio: 2.63
split 1 | lgbm_baseline_consumption_business | train time: 27.00s
Train MAE: 62.75 | Test MAE: 82.55 | Ratio: 1.32
split 2 | lgbm_baseline_production_not_business | train time: 13.66s
Train 

In [ ]:
# q = df.copy()
# q["year"] = q["datetime"].dt.year
# q.groupby(
#     ["year", "month", "is_business", "is_consumption"],
#     as_index=False,
# )["target"].agg(["mean", "std", "median"])

In [ ]:
actual_iters
# [{'split': 0, 'is_bus': 0, 'is_cons': 0, 'best_iter': 1500},
#  {'split': 0, 'is_bus': 1, 'is_cons': 0, 'best_iter': 1469},
#  {'split': 0, 'is_bus': 0, 'is_cons': 1, 'best_iter': 510},
#  {'split': 0, 'is_bus': 1, 'is_cons': 1, 'best_iter': 911},
#  {'split': 1, 'is_bus': 0, 'is_cons': 0, 'best_iter': 486},
#  {'split': 1, 'is_bus': 1, 'is_cons': 0, 'best_iter': 971},
#  {'split': 1, 'is_bus': 0, 'is_cons': 1, 'best_iter': 1450},
#  {'split': 1, 'is_bus': 1, 'is_cons': 1, 'best_iter': 1498},
#  {'split': 2, 'is_bus': 0, 'is_cons': 0, 'best_iter': 1314},
#  {'split': 2, 'is_bus': 1, 'is_cons': 0, 'best_iter': 846},
#  {'split': 2, 'is_bus': 0, 'is_cons': 1, 'best_iter': 813},
#  {'split': 2, 'is_bus': 1, 'is_cons': 1, 'best_iter': 659}]

[{'split': 0, 'is_bus': 0, 'is_cons': 0, 'best_iter': 59},
 {'split': 0, 'is_bus': 1, 'is_cons': 0, 'best_iter': 149},
 {'split': 0, 'is_bus': 0, 'is_cons': 1, 'best_iter': 1000},
 {'split': 0, 'is_bus': 1, 'is_cons': 1, 'best_iter': 80},
 {'split': 1, 'is_bus': 0, 'is_cons': 0, 'best_iter': 447},
 {'split': 1, 'is_bus': 1, 'is_cons': 0, 'best_iter': 38},
 {'split': 1, 'is_bus': 0, 'is_cons': 1, 'best_iter': 995},
 {'split': 1, 'is_bus': 1, 'is_cons': 1, 'best_iter': 889},
 {'split': 2, 'is_bus': 0, 'is_cons': 0, 'best_iter': 222},
 {'split': 2, 'is_bus': 1, 'is_cons': 0, 'best_iter': 202},
 {'split': 2, 'is_bus': 0, 'is_cons': 1, 'best_iter': 1000},
 {'split': 2, 'is_bus': 1, 'is_cons': 1, 'best_iter': 383}]

In [ ]:
# sns.scatterplot(
#     baseline_ev_his[2]["lgbm_baseline_consumption_not_business"]["valid_0"][
#         "l1"
#     ]
# )

## Results
### Scores

In [ ]:
baseline_scores_df = pd.DataFrame(baseline_scores)
baseline_scores_df["ratio"] = (
    baseline_scores_df["test_mae"] / baseline_scores_df["train_mae"]
)
baseline_scores_df.sort_values(["is_business", "is_consumption", "split"])

,split,is_business,is_consumption,train_mae,test_mae,ratio
3,0,business,consumption,89.35,102.93,1.15
7,1,business,consumption,62.75,82.55,1.32
11,2,business,consumption,72.39,117.82,1.63
1,0,business,production,27.44,3.32,0.12
5,1,business,production,30.63,9.81,0.32
9,2,business,production,24.75,32.50,1.31
2,0,not_business,consumption,8.96,26.59,2.97
6,1,not_business,consumption,9.46,24.91,2.63
10,2,not_business,consumption,9.86,33.69,3.42
0,0,not_business,production,27.53,4.17,0.15


In [ ]:
# split	is_business	is_consumption	train_mae	test_mae	ratio
# 3	0	business	consumption	57.70	92.28	1.60
# 7	1	business	consumption	59.21	83.88	1.42
# 11	2	business	consumption	80.28	119.13	1.48
# 1	0	business	production	25.13	3.18	0.13
# 5	1	business	production	25.79	9.40	0.36
# 9	2	business	production	36.33	37.06	1.02
# 2	0	not_business	consumption	8.14	26.09	3.21
# 6	1	not_business	consumption	8.58	24.31	2.83
# 10	2	not_business	consumption	8.93	33.83	3.79
# 0	0	not_business	production	26.34	4.16	0.16
# 4	1	not_business	production	26.38	17.63	0.67
# 8	2	not_business	production	24.20	50.37	2.08

In [ ]:
# scores = baseline_scores_df.copy()

### Feature Importance

In [ ]:
# fi_df = pd.concat(baseline_fi)
# fi_df

In [ ]:
# fi_melted = (
#     fi_df.drop(columns=["split"])
#     .groupby(["segment"], as_index=False, sort=False)
#     .mean()
#     .set_index("segment")
#     .rename_axis("")
#     .T
# )
# # fi_melted

In [ ]:
# # head/tail features per segment
# for col in fi_melted.columns:
#     print(
#         col,
#         "\n\nhead",
#         fi_melted[col].sort_values(ascending=False).head(5),
#         "\n\ntail",
#         fi_melted[col].sort_values(ascending=False).tail(5),
#         "\n",
#         sep="",
#     )

### Residuals

In [ ]:
# preds_df = pd.concat(baseline_preds)

In [ ]:
# preds_df["hour"] = preds_df.index.map(df["datetime"].dt.hour)
# # preds_df["month"] = preds_df.index.map(df["datetime"].dt.month)
# preds_df["dayofweek"] = preds_df.index.map(df["datetime"].dt.dayofweek)
# preds_df["date"] = preds_df.index.map(df["datetime"])

In [ ]:
# preds_df_dated = preds_df.set_index("date")
# # preds_df_dated["abs_e"].resample("W").mean().plot()

In [ ]:
# fig, axes = plt.subplots(2, 2, figsize=(14, 8))
# segments = preds_df_dated.groupby(["is_consumption", "is_business"])

# for ax, ((is_cons, is_bus), group) in zip(axes.flatten(), segments):
#     group["abs_e"].resample("W").mean().plot(ax=ax)
#     ax.set_title(
#         f"{'consumption' if is_cons else 'production'} | "
#         f"{'business' if is_bus else 'not_business'}"
#     )
# plt.tight_layout()

In [ ]:
# fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# for ax, col in zip(axes, ["hour", "dayofweek"]):
#     preds_df.groupby(col)["abs_e"].mean().plot(ax=ax, title=f"MAE by {col}")
# plt.tight_layout()

In [ ]:
# county_errors = (
#     preds_df.join(df[["county"]])
#     .groupby(
#         ["county", "is_consumption", "is_business"],
#         as_index=False,
#         sort=False,
#     )["abs_e"]
#     .mean()
#     # .reset_index()
#     .sort_values("abs_e", ascending=False)
# )
# display(county_errors.head(10))

## prod target transform

In [ ]:
SEGMENT_F

['county', 'product_type', 'is_business']

In [ ]:
df["installed_capacity"] = (
    df.groupby(SEGMENT_F)["installed_capacity"].bfill().ffill().fillna(1.0)
)

In [ ]:
prod_mask = df["is_consumption"] == 0

In [ ]:
df.loc[prod_mask, "target_yield"] = df.loc[prod_mask, "target"] / df.loc[
    prod_mask, "installed_capacity"
].clip(lower=1.0)

In [ ]:
yield_preds = []
yield_scores = []
yield_ev_his = defaultdict(dict)
yield_fi = []
yield_iters = []

for i, split in enumerate(splits_val):
    is_cons = 0
    is_consumption_label = "consumption" if is_cons else "production"

    for is_bus in df["is_business"].unique():
        is_business_label = "business" if is_bus else "not_business"
        purpose = (
            f"lgbm_target_capacity_{is_consumption_label}_{is_business_label}"
        )

        df_subset = df[
            (df["is_consumption"] == is_cons) & (df["is_business"] == is_bus)
        ]
        test_mask = df_subset["datetime"].between(
            split["test"][0], split["test"][1]
        )
        train_cols = df_subset.columns.difference(
            COLUMNS_TO_DROP + ["target", "target_yield"], sort=False
        )

        X_test = df_subset.loc[test_mask, train_cols]
        y_test = df_subset.loc[test_mask, "target"]
        capacity_test = df_subset.loc[test_mask, "installed_capacity"]
        config = configs[(is_bus, is_cons)]

        start_time = time.perf_counter()

        model, was_trained, history = load_or_train_sklearn(
            models_dir="../models",
            notebook="feature_engineering",
            purpose=purpose,
            model_label=config["model_label"],
            model_cls=config["model_cls"],
            model_params=config["model_params"],
            split=split,
            df=df_subset,
            target_col="target_yield",
            drop_cols=COLUMNS_TO_DROP + ["target"],
            cat_cols=cat_cols,
            eval_week=config["eval_week"],
            early_stopping_rounds=config["early_stopping_rounds"],
            random_state=RAND,
        )
        preds_yield = model.predict(X_test)
        preds_final = preds_yield * capacity_test.values
        test_mae = MAE(y_test, preds_final)

        elapsed_time = time.perf_counter() - start_time
        print(
            f"split {i}",
            f"{purpose}",
            # f"{model_label}",
            f"train time: {elapsed_time:.2f}s",
            sep=" | ",
        )
        yield_ev_his[i][purpose] = history

        train_mask = df_subset["datetime"].between(
            split["train"][0],
            (
                (split["train"][1] - pd.Timedelta("1W"))
                if config["early_stopping_rounds"]
                else split["train"][1]
            ),
        )  # exclude eval period
        X_train = df_subset.loc[train_mask, train_cols]
        y_train = df_subset.loc[train_mask, "target"]
        capacity_train = df_subset.loc[train_mask, "installed_capacity"]

        train_preds_yield = model.predict(X_train)
        train_preds_final = train_preds_yield * capacity_train.values
        train_mae = MAE(y_train, train_preds_final)

        yield_scores.append(
            {
                "split": i,
                "is_business": is_business_label,
                "train_mae": train_mae,
                "test_mae": test_mae,
            }
        )
        print(
            f"Train MAE: {train_mae:.2f}",
            f"Test MAE: {test_mae:.2f}",
            f"Ratio: {test_mae / train_mae:.2f}",
            sep=" | ",
        )

        fold_df = pd.DataFrame(
            {
                "split": i,
                "is_business": X_test["is_business"],
                "target": y_test,
                "preds": preds_final,
            },
            index=X_test.index,
        )
        fold_df["abs_e"] = (fold_df["target"] - fold_df["preds"]).abs()
        yield_preds.append(fold_df)

        meta = pd.DataFrame(
            [[i, f"{is_business_label}_{is_consumption_label}"]],
            columns=["split", "segment"],
        )
        fi = pd.DataFrame(
            [model.booster_.feature_importance(importance_type="gain")],
            columns=model.feature_name_,
        ).drop(columns=["is_business", "is_consumption"])
        fi = pd.concat([meta, fi], axis=1)
        yield_fi.append(fi)

        yield_iters.append(
            {
                "split": i,
                "is_bus": is_bus,
                "best_iter": model.booster_.best_iteration,
            }
        )

        # display()

split 0 | lgbm_target_capacity_production_not_business | train time: 16.92s
Train MAE: 20.06 | Test MAE: 4.20 | Ratio: 0.21
split 0 | lgbm_target_capacity_production_business | train time: 10.32s
Train MAE: 23.13 | Test MAE: 3.38 | Ratio: 0.15
split 1 | lgbm_target_capacity_production_not_business | train time: 16.11s
Train MAE: 18.92 | Test MAE: 16.91 | Ratio: 0.89
split 1 | lgbm_target_capacity_production_business | train time: 13.64s
Train MAE: 37.91 | Test MAE: 10.25 | Ratio: 0.27
split 2 | lgbm_target_capacity_production_not_business | train time: 6.40s
Train MAE: 37.29 | Test MAE: 49.81 | Ratio: 1.34
split 2 | lgbm_target_capacity_production_business | train time: 9.60s
Train MAE: 39.28 | Test MAE: 33.84 | Ratio: 0.86


In [ ]:
yield_iters
# [{'split': 0, 'is_bus': 0, 'best_iter': 158},
#  {'split': 0, 'is_bus': 1, 'best_iter': 26},
#  {'split': 1, 'is_bus': 0, 'best_iter': 135},
#  {'split': 1, 'is_bus': 1, 'best_iter': 19},
#  {'split': 2, 'is_bus': 0, 'best_iter': 20},
#  {'split': 2, 'is_bus': 1, 'best_iter': 16}]

[{'split': 0, 'is_bus': 0, 'best_iter': 332},
 {'split': 0, 'is_bus': 1, 'best_iter': 94},
 {'split': 1, 'is_bus': 0, 'best_iter': 269},
 {'split': 1, 'is_bus': 1, 'best_iter': 8},
 {'split': 2, 'is_bus': 0, 'best_iter': 10},
 {'split': 2, 'is_bus': 1, 'best_iter': 7}]

In [ ]:
yield_scores_df = pd.DataFrame(yield_scores)
yield_scores_df["ratio"] = (
    yield_scores_df["test_mae"] / yield_scores_df["train_mae"]
)
yield_scores_df.sort_values(["is_business", "split"])

# 	split	is_business	train_mae	test_mae	ratio
# 1	0	business	31.16	3.17	0.10
# 3	1	business	34.28	9.53	0.28
# 5	2	business	35.99	32.54	0.90
# 0	0	not_business	19.42	3.76	0.19
# 2	1	not_business	18.62	15.97	0.86
# 4	2	not_business	37.08	49.50	1.33

,split,is_business,train_mae,test_mae,ratio
1,0,business,23.13,3.38,0.15
3,1,business,37.91,10.25,0.27
5,2,business,39.28,33.84,0.86
0,0,not_business,20.06,4.20,0.21
2,1,not_business,18.92,16.91,0.89
4,2,not_business,37.29,49.81,1.34


In [ ]:
baseline_scores_df[
    baseline_scores_df["is_consumption"] == "production"
].sort_values(["is_business", "split"])

,split,is_business,is_consumption,train_mae,test_mae,ratio
1,0,business,production,27.44,3.32,0.12
5,1,business,production,30.63,9.81,0.32
9,2,business,production,24.75,32.50,1.31
0,0,not_business,production,27.53,4.17,0.15
4,1,not_business,production,24.81,17.47,0.70
8,2,not_business,production,24.75,51.53,2.08


# 4. New Features

hypothetical areas:
- solar astronomy
- additional target lags
- Rolling statistics
- Weather interactions
- client features
- price features
- calendar and time patterns
- in-county additional weather statistics (std)
- target encoding / segment aggregations
- target transformation

#### Target Lags

In [ ]:
dp_test = DataPipeline(PARQUET_DATA_PATH)

In [ ]:
dp_test.load()

In [ ]:
dp_test.prepare()

In [ ]:
dp_test.merge()

In [ ]:
dp_test.transform_datetime()

In [ ]:
target_lags = {
    "2d": {"24h": ["mean", "std"], "168h": ["mean", "std"]},
    "49h": None,
    "50h": None,
    "71h": None,
    "73h": None,
    "7d": {"24h": ["mean", "std"]},
    "167h": None,
    "169h": None,
    "14d": {"24h": ["mean", "std"]},
    "335h": None,
    "337h": None,
    "365d": {"24h": ["mean"], "6h": ["mean"]},
}

In [ ]:
dp_test.add_lags_and_rollings("target", target_lags, "datetime", CATEGORICAL_F)

In [ ]:
dp_test.add_ratio_features()

In [ ]:
df = dp_test.df

In [ ]:
df.info(verbose=True, show_counts=True)

In [ ]:
target_lags_preds = []
target_lags_scores = []
# target_lags_ev_his = defaultdict(dict)
target_lags_fi = []

for i, split in enumerate(splits_val):
    for is_cons in df["is_consumption"].unique():
        is_consumption_label = "consumption" if is_cons else "production"

        for is_bus in df["is_business"].unique():
            is_business_label = "business" if is_bus else "not_business"
            purpose = f"target_lags_{is_consumption_label}_{is_business_label}"

            df_subset = df[
                (df["is_consumption"] == is_cons)
                & (df["is_business"] == is_bus)
            ]
            test_mask = df_subset["datetime"].between(
                split["test"][0], split["test"][1]
            )

            train_cols = df_subset.columns.difference(
                COLUMNS_TO_DROP + ["target"], sort=False
            )

            X_test = df_subset.loc[test_mask, train_cols]
            y_test = df_subset.loc[test_mask, "target"]

            if not (is_cons and is_bus):
                model_label, model_cls, model_params = configs[0]  # mae
            else:
                model_label, model_cls, model_params = configs[1]  # mse

            start_time = time.perf_counter()

            model, was_trained, history = load_or_train_sklearn(
                models_dir="../models",
                notebook="feature_engineering",
                purpose=purpose,
                model_label=model_label,
                model_cls=model_cls,
                model_params=model_params,
                split=split,
                df=df_subset,
                target_col="target",
                drop_cols=COLUMNS_TO_DROP,
                cat_cols=cat_cols,
                eval_week=eval_week,
                early_stopping_rounds=early_stopping_rounds,
                random_state=RAND,
            )

            elapsed_time = time.perf_counter() - start_time
            print(
                f"split {i}",
                f"{purpose}",
                # f"{model_label}",
                f"train time: {elapsed_time:.2f}s",
                sep=" | ",
            )

            # target_lags_ev_his[i][purpose] = history
            test_preds = model.predict(X_test)
            test_mae = MAE(y_test, test_preds)

            train_mask = df_subset["datetime"].between(
                split["train"][0],
                (
                    (split["train"][1] - pd.Timedelta("1W"))
                    if early_stopping_rounds
                    else split["train"][1]
                ),
            )  # exclude eval period
            X_train = df_subset.loc[train_mask, train_cols]
            y_train = df_subset.loc[train_mask, "target"]
            train_preds = model.predict(X_train)
            train_mae = MAE(y_train, train_preds)

            target_lags_scores.append(
                {
                    "split": i,
                    "purpose": purpose,
                    # "is_business": is_business_label,
                    # "is_consumption": is_consumption_label,
                    "train_mae": train_mae,
                    "test_mae": test_mae,
                }
            )
            print(
                f"Train MAE: {train_mae:.2f}",
                f"Test MAE: {test_mae:.2f}",
                f"Ratio: {test_mae / train_mae:.2f}",
                sep=" | ",
            )

            fold_df = pd.DataFrame(
                {
                    # "model_label": model_label,
                    "split": i,
                    "is_business": X_test["is_business"],
                    "is_consumption": X_test["is_consumption"],
                    "target": y_test,
                    "preds": test_preds,
                },
                index=X_test.index,
            )
            fold_df["abs_e"] = (fold_df["target"] - fold_df["preds"]).abs()
            target_lags_preds.append(fold_df)

            fi = pd.DataFrame(
                {
                    "feature": model.feature_name_,
                    "importance_gain": model.booster_.feature_importance(
                        importance_type="gain"
                    ),
                }
            )
            fi = fi.set_index("feature").T
            fi.columns.name = ""
            fi["split"] = i
            fi["purpose"] = purpose
            target_lags_fi.append(fi)

In [ ]:
# split 0 | lgbm_baseline_production_not_business | train time: 40.83s
# Train MAE: 18.35 | Test MAE: 3.83 | Ratio: 0.21
# split 0 | lgbm_baseline_production_business | train time: 30.05s
# Train MAE: 19.26 | Test MAE: 3.26 | Ratio: 0.17
# split 0 | lgbm_baseline_consumption_not_business | train time: 27.91s
# Train MAE: 7.00 | Test MAE: 26.25 | Ratio: 3.75
# split 0 | lgbm_baseline_consumption_business | train time: 21.71s
# Train MAE: 43.49 | Test MAE: 93.79 | Ratio: 2.16
# split 1 | lgbm_baseline_production_not_business | train time: 30.37s
# Train MAE: 17.68 | Test MAE: 15.67 | Ratio: 0.89
# split 1 | lgbm_baseline_production_business | train time: 34.76s
# Train MAE: 18.89 | Test MAE: 9.07 | Ratio: 0.48
# split 1 | lgbm_baseline_consumption_not_business | train time: 30.88s
# Train MAE: 7.24 | Test MAE: 23.69 | Ratio: 3.27
# split 1 | lgbm_baseline_consumption_business | train time: 24.29s
# Train MAE: 44.22 | Test MAE: 83.69 | Ratio: 1.89
# split 2 | lgbm_baseline_production_not_business | train time: 30.72s
# Train MAE: 17.68 | Test MAE: 44.44 | Ratio: 2.51
# split 2 | lgbm_baseline_production_business | train time: 44.22s
# Train MAE: 18.80 | Test MAE: 31.66 | Ratio: 1.68
# split 2 | lgbm_baseline_consumption_not_business | train time: 27.83s
# Train MAE: 7.60 | Test MAE: 33.14 | Ratio: 4.36
# split 2 | lgbm_baseline_consumption_business | train time: 22.90s
# Train MAE: 44.88 | Test MAE: 112.55 | Ratio: 2.51

In [ ]:
target_lags_scores_df = pd.DataFrame(target_lags_scores)
target_lags_scores_df["ratio"] = (
    target_lags_scores_df["test_mae"] / target_lags_scores_df["train_mae"]
)

In [ ]:
target_lags_scores_df

In [ ]:
baseline_scores_df.merge(
    target_lags_scores_df, how="left", on=["split", "purpose"]
)